<a href="https://colab.research.google.com/github/EsatovinVictoria/ML_Semester5_EsatovinEbenhaezerVictoria/blob/main/Pertemuan_6/Praktikum_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install hnswlib

import hnswlib
import numpy as np
import time
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Definisikan parameter eksperimen
# Mencakup kombinasi: Data Size (1000, 1000000), Dimensi (2, 5), Metrik (L2, Cosine)
# Catatan: Untuk 1.000.000 data, Exact NN (Brute Force) bisa memakan waktu cukup lama,
# namun tetap layak dijalankan untuk menunjukkan perbedaan performa yang signifikan.

configs = [
    {"n_elements": 1000, "dim": 2, "space": "l2", "metric_sklearn": "euclidean"},
    {"n_elements": 1000, "dim": 2, "space": "cosine", "metric_sklearn": "cosine"},
    {"n_elements": 1000, "dim": 5, "space": "l2", "metric_sklearn": "euclidean"},
    {"n_elements": 1000, "dim": 5, "space": "cosine", "metric_sklearn": "cosine"},
    {"n_elements": 1000000, "dim": 2, "space": "l2", "metric_sklearn": "euclidean"},
    {"n_elements": 1000000, "dim": 2, "space": "cosine", "metric_sklearn": "cosine"},
    {"n_elements": 1000000, "dim": 5, "space": "l2", "metric_sklearn": "euclidean"},
    {"n_elements": 1000000, "dim": 5, "space": "cosine", "metric_sklearn": "cosine"}
]

k = 5
results = []

print("Memulai eksperimen perbandingan...")

for cfg in configs:
    n_elements = cfg["n_elements"]
    dim = cfg["dim"]
    space = cfg["space"]
    metric_sklearn = cfg["metric_sklearn"]

    print(f"Menjalankan eksperimen: N={n_elements}, Dim={dim}, Metrik={space}...")

    # Generate random data
    data = np.random.random((n_elements, dim)).astype(np.float32)
    query = np.random.random((1, dim)).astype(np.float32)

    # 1. Exact NN (Sklearn Brute Force)
    try:
        nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=metric_sklearn)
        nn.fit(data)
        start = time.time()
        distances_exact, indices_exact = nn.kneighbors(query)
        time_exact = time.time() - start
    except Exception as e:
        time_exact = None
        print(f"Gagal menjalankan Exact NN untuk N={n_elements}: {e}")

    # 2. HNSW
    try:
        # Inisialisasi index HNSW
        p = hnswlib.Index(space=space, dim=dim)
        p.init_index(max_elements=n_elements, ef_construction=100, M=16)
        p.add_items(data)
        p.set_ef(50)

        start = time.time()
        labels_hnsw, distances_hnsw = p.knn_query(query, k=k)
        time_hnsw = time.time() - start
    except Exception as e:
        time_hnsw = None
        print(f"Gagal menjalankan HNSW untuk N={n_elements}: {e}")

    # Hitung rasio kecepatan
    speedup = time_exact / time_hnsw if (time_exact and time_hnsw) else None

    results.append({
        "Jumlah Data": n_elements,
        "Dimensi": dim,
        "Metrik": space,
        "Waktu Exact NN (detik)": f"{time_exact:.6f}" if time_exact is not None else "N/A",
        "Waktu HNSW (detik)": f"{time_hnsw:.6f}" if time_hnsw is not None else "N/A",
        "Speedup (Exact / HNSW)": f"{speedup:.2f}x" if speedup is not None else "N/A"
    })

# Tampilkan tabel hasil
df_results = pd.DataFrame(results)
import IPython
IPython.display.display(df_results)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679797 sha256=57f436f617a6262472b57b23c1f53bc8344688841551bf9bbf3a8f04c43fe982
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib
Memulai eksperimen perbandingan...
Menjalankan eksperimen: N=1000, Dim=2, Metrik=l2...
Menjalankan eksperimen: N=1000, Dim=2, Metrik=cosine...
Menjalankan eksperimen: N=1000, Dim=5, Metrik=l2...
Menjalankan eksperimen: N=1000, Dim=5, Metrik=cosine...
Menjalankan eksperimen: N=1000000, Dim=2, Metrik=l2...
Menjalankan eksperimen: N=1000000, Dim=2, Metrik=cosine...
Menjalankan eksperimen: N=1000000, Dim=5, Metrik=l2...
Menjalankan eksperimen: N=1000000, Dim=5, Metrik=cosine...


,Jumlah Data,Dimensi,Metrik,Waktu Exact NN (detik),Waktu HNSW (detik),Speedup (Exact / HNSW)
0,1000,2,l2,0.037023,0.000081,459.43x
1,1000,2,cosine,0.021983,0.000077,286.34x
2,1000,5,l2,0.000698,0.000084,8.29x
3,1000,5,cosine,0.001481,0.000071,20.98x
4,1000000,2,l2,0.046303,0.000148,312.23x
5,1000000,2,cosine,0.084434,0.000156,542.33x
6,1000000,5,l2,0.031981,0.000189,169.37x
7,1000000,5,cosine,0.084899,0.000135,628.03x
